# 🔬 Decode Lab Internship — Week 2
## E-Commerce Data Analytics Project
### `decode_lab_internship_w2`

**Dataset:** `Dataset_for_Data_Analytics.xlsx`  
**Period:** January 2023 – June 2025  
**Records:** 1,200 orders × 14 features  

---

### Project Roadmap
| Phase | Goal | Key Skills |
|-------|------|-----------|
| **1 — Load & Understand** | Explore structure and schema | Data understanding, exploratory thinking |
| **2 — EDA** | Find patterns, trends, outliers | Statistical thinking, data analysis |
| **3 — Visualize** | Communicate insights with charts | Data visualization, storytelling |
| **4 — Predictive Models** | Regression + classification | Basic machine learning |
| **5 — Insights** | Summarize findings & recommendations | Business insight |


## Phase 1 — Load & Understand the Dataset

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Import all required libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#0f1117',
    'axes.edgecolor':   '#2e3347',
    'axes.labelcolor':  '#c8ccd8',
    'xtick.color':      '#6b738a',
    'ytick.color':      '#6b738a',
    'text.color':       '#c8ccd8',
    'grid.color':       '#1e2233',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.titlepad':    12,
    'figure.dpi':       110,
})

ACCENT   = '#00e5a0'
ACCENT2  = '#7c6fff'
ACCENT3  = '#ff6b6b'
ACCENT4  = '#ffb84d'
PALETTE  = [ACCENT, ACCENT2, ACCENT4, ACCENT3, '#4d9fff', '#e05fe0', '#5fd4e0']

print("✅ All libraries imported successfully.")


In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Load the dataset
# ─────────────────────────────────────────────

# Update this path to wherever your file lives
FILE_PATH = 'Dataset_for_Data_Analytics.xlsx'

df = pd.read_excel(FILE_PATH)

# Rename dataset variable as required by project spec
decode_lab_internship_w2 = df.copy()

print(f"✅ Dataset loaded as: decode_lab_internship_w2")
print(f"   Shape : {decode_lab_internship_w2.shape[0]:,} rows × {decode_lab_internship_w2.shape[1]} columns")
print(f"   Memory: {decode_lab_internship_w2.memory_usage(deep=True).sum() / 1024:.1f} KB")


In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Understand structure — columns & dtypes
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

print("=" * 60)
print("COLUMN OVERVIEW")
print("=" * 60)

schema = pd.DataFrame({
    'Column':      df.columns,
    'Dtype':       df.dtypes.values,
    'Non-Null':    df.notnull().sum().values,
    'Null Count':  df.isnull().sum().values,
    'Unique':      df.nunique().values,
    'Sample':      [str(df[c].dropna().iloc[0])[:30] for c in df.columns]
})
print(schema.to_string(index=False))

print()
print(f"Date range: {df['Date'].min().date()}  →  {df['Date'].max().date()}")
print(f"Products  : {sorted(df['Product'].unique())}")
print(f"Statuses  : {sorted(df['OrderStatus'].unique())}")
print(f"Payment   : {sorted(df['PaymentMethod'].unique())}")
print(f"Channels  : {sorted(df['ReferralSource'].unique())}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Dataset size, features & descriptive stats
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

print("NUMERICAL FEATURES — DESCRIPTIVE STATISTICS")
print("=" * 60)
print(df[['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']].describe().round(2))

print()
print("KEY BUSINESS METRICS")
print("=" * 60)
print(f"  Total Revenue      : ${df['TotalPrice'].sum():,.2f}")
print(f"  Average Order Value: ${df['TotalPrice'].mean():,.2f}")
print(f"  Median Order Value : ${df['TotalPrice'].median():,.2f}")
print(f"  Max Single Order   : ${df['TotalPrice'].max():,.2f}")
print(f"  Min Single Order   : ${df['TotalPrice'].min():,.2f}")
print()
print(f"  Orders WITH coupon : {df['CouponCode'].notna().sum()} ({df['CouponCode'].notna().mean()*100:.1f}%)")
print(f"  Orders without     : {df['CouponCode'].isna().sum()}  ({df['CouponCode'].isna().mean()*100:.1f}%)")


---
## Phase 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Check missing values & duplicates
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

print("MISSING VALUES")
print("=" * 40)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percent (%)': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

print()
print(f"Duplicate rows : {df.duplicated().sum()}")
print()
print("NOTE: CouponCode has 309 NaN values — expected (orders without a coupon).")
print("      All other columns are 100% complete. No cleaning needed.")


In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Value counts for categorical columns
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

cat_cols = ['Product', 'OrderStatus', 'PaymentMethod', 'ReferralSource']

for col in cat_cols:
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    summary = pd.DataFrame({'Count': vc, 'Percent (%)': pct})
    print(f"\n{'='*40}")
    print(f" {col}")
    print(f"{'='*40}")
    print(summary.to_string())


In [ ]:
# ─────────────────────────────────────────────
# CELL 7: Outlier detection — IQR method
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

num_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']

print("OUTLIER DETECTION (IQR Method)")
print("=" * 50)
for col in num_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"  {col:<14}  bounds: [{lower:.2f}, {upper:.2f}]  →  {len(outliers)} outliers")


In [ ]:
# ─────────────────────────────────────────────
# CELL 8: Revenue breakdowns by category
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

# By Product
prod_rev = df.groupby('Product')['TotalPrice'].agg(
    Orders='count', TotalRevenue='sum', AvgOrder='mean'
).sort_values('TotalRevenue', ascending=False).round(2)
print("REVENUE BY PRODUCT")
print(prod_rev.to_string())

# By Status
print("\nREVENUE BY ORDER STATUS")
status_rev = df.groupby('OrderStatus')['TotalPrice'].agg(
    Orders='count', TotalRevenue='sum', AvgOrder='mean'
).round(2)
print(status_rev.to_string())

# Coupon impact
has_coupon    = df[df['CouponCode'].notna()]['TotalPrice'].mean()
no_coupon     = df[df['CouponCode'].isna()]['TotalPrice'].mean()
print(f"\nCOUPON IMPACT ON AVG ORDER VALUE")
print(f"  With coupon   : ${has_coupon:.2f}")
print(f"  Without coupon: ${no_coupon:.2f}")
print(f"  Difference    : ${has_coupon - no_coupon:.2f}")


In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Correlation matrix
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

corr = df[['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']].corr().round(3)

fig, ax = plt.subplots(figsize=(6, 4.5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # show lower triangle
sns.heatmap(
    corr, annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, linewidths=0.5, linecolor='#1e2233',
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Correlation Matrix — Numerical Features')
plt.tight_layout()
plt.show()

print("\nKEY CORRELATIONS:")
print(f"  UnitPrice  ↔ TotalPrice  : {corr.loc['UnitPrice','TotalPrice']:.3f}  (strong positive)")
print(f"  Quantity   ↔ TotalPrice  : {corr.loc['Quantity','TotalPrice']:.3f}")
print(f"  ItemsInCart↔ TotalPrice  : {corr.loc['ItemsInCart','TotalPrice']:.3f}")


---
## Phase 3 — Data Visualizations

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Monthly revenue trend (line chart)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2.copy()
df['Month'] = df['Date'].dt.to_period('M')

monthly = df.groupby('Month')['TotalPrice'].sum().reset_index()
monthly['Month_dt'] = monthly['Month'].dt.to_timestamp()
monthly['MA3'] = monthly['TotalPrice'].rolling(3, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(monthly['Month_dt'], monthly['TotalPrice'],
                alpha=0.15, color=ACCENT)
ax.plot(monthly['Month_dt'], monthly['TotalPrice'],
        color=ACCENT, linewidth=1.5, label='Monthly Revenue', marker='o', markersize=3)
ax.plot(monthly['Month_dt'], monthly['MA3'],
        color=ACCENT4, linewidth=2.5, linestyle='--', label='3-Month Moving Avg')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Monthly Revenue Trend  |  Jan 2023 – Jun 2025')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
ax.legend(framealpha=0.1, edgecolor='#2e3347')
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 11: Revenue by product (horizontal bar)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

prod = df.groupby('Product')['TotalPrice'].sum().sort_values()
colors = [ACCENT3 if v == prod.min() else ACCENT if v == prod.max() else ACCENT2
          for v in prod.values]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(prod.index, prod.values, color=colors, edgecolor='none', height=0.6)

for bar in bars:
    ax.text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width()/1000:.1f}K', va='center', fontsize=9, color='#c8ccd8')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.set_title('Total Revenue by Product Category')
ax.set_xlabel('Total Revenue')
ax.set_xlim(0, prod.max() * 1.15)
ax.grid(True, axis='x')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 12: Order status & payment method (side by side)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Doughnut — Order Status
status_counts = df['OrderStatus'].value_counts()
wedges, texts, autotexts = axes[0].pie(
    status_counts.values,
    labels=status_counts.index,
    autopct='%1.1f%%',
    colors=[ACCENT3, ACCENT4, ACCENT2, '#4d9fff', ACCENT],
    startangle=90,
    wedgeprops={'width': 0.55, 'edgecolor': '#0f1117', 'linewidth': 2},
    pctdistance=0.75
)
for t in autotexts:
    t.set_fontsize(9)
    t.set_color('white')
axes[0].set_title('Order Status Distribution')

# Bar — Payment Method
pay_counts = df['PaymentMethod'].value_counts()
axes[1].bar(pay_counts.index, pay_counts.values,
            color=PALETTE[:len(pay_counts)], edgecolor='none', width=0.6)
for i, v in enumerate(pay_counts.values):
    axes[1].text(i, v + 3, str(v), ha='center', fontsize=9)
axes[1].set_title('Orders by Payment Method')
axes[1].set_ylabel('Number of Orders')
axes[1].set_ylim(0, pay_counts.max() * 1.12)
axes[1].grid(True, axis='y')

plt.suptitle('Order Status & Payment Method', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 13: Referral source & avg order by channel
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

ref_orders = df['ReferralSource'].value_counts()
ref_avg    = df.groupby('ReferralSource')['TotalPrice'].mean().reindex(ref_orders.index)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Orders per channel
axes[0].bar(ref_orders.index, ref_orders.values,
            color=PALETTE[:len(ref_orders)], edgecolor='none', width=0.6)
for i, v in enumerate(ref_orders.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=9)
axes[0].set_title('Orders by Referral Source')
axes[0].set_ylabel('Order Count')
axes[0].set_ylim(0, ref_orders.max() * 1.12)
axes[0].grid(True, axis='y')

# Avg order per channel
bars = axes[1].bar(ref_avg.index, ref_avg.values,
                   color=PALETTE[:len(ref_avg)], edgecolor='none', width=0.6)
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'${bar.get_height():.0f}', ha='center', fontsize=9)
axes[1].set_title('Avg Order Value by Referral Source')
axes[1].set_ylabel('Avg Order Value ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))
axes[1].set_ylim(0, ref_avg.max() * 1.15)
axes[1].grid(True, axis='y')

plt.suptitle('Referral Source Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 14: TotalPrice distribution (histogram + KDE)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram + KDE
axes[0].hist(df['TotalPrice'], bins=40, color=ACCENT, alpha=0.7, edgecolor='none')
axes[0].axvline(df['TotalPrice'].mean(),   color=ACCENT4, linestyle='--', linewidth=1.8, label=f"Mean  ${df['TotalPrice'].mean():.0f}")
axes[0].axvline(df['TotalPrice'].median(), color=ACCENT3, linestyle='--', linewidth=1.8, label=f"Median ${df['TotalPrice'].median():.0f}")
axes[0].set_title('Distribution of Total Order Value')
axes[0].set_xlabel('Total Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend(framealpha=0.1, edgecolor='#2e3347')
axes[0].grid(True, axis='y')

# Boxplot by product
products = df['Product'].unique()
data_by_product = [df[df['Product'] == p]['TotalPrice'].values for p in sorted(products)]
bp = axes[1].boxplot(data_by_product, labels=sorted(products),
                     patch_artist=True, notch=False,
                     medianprops=dict(color=ACCENT4, linewidth=2),
                     whiskerprops=dict(color='#6b738a'),
                     capprops=dict(color='#6b738a'),
                     flierprops=dict(marker='o', color=ACCENT3, markersize=3, alpha=0.5))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Order Value Distribution by Product')
axes[1].set_xlabel('Product')
axes[1].set_ylabel('Total Price ($)')
axes[1].grid(True, axis='y')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Revenue Distribution Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 15: Yearly revenue comparison (bar)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2.copy()
df['Year'] = df['Date'].dt.year

yearly = df.groupby('Year').agg(
    Revenue=('TotalPrice', 'sum'),
    Orders=('OrderID', 'count'),
    AvgOrder=('TotalPrice', 'mean')
).round(2)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))

colors_yr = [ACCENT, ACCENT2, ACCENT4]
for i, (metric, label, fmt) in enumerate([
    ('Revenue',  'Total Revenue ($)',     lambda x, _: f'${x/1000:.0f}K'),
    ('Orders',   'Number of Orders',      lambda x, _: str(int(x))),
    ('AvgOrder', 'Avg Order Value ($)',   lambda x, _: f'${x:.0f}'),
]):
    bars = axes[i].bar(yearly.index.astype(str), yearly[metric],
                       color=colors_yr, edgecolor='none', width=0.5)
    for bar in bars:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                     fmt(bar.get_height(), None), ha='center', fontsize=9)
    axes[i].set_title(label)
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
    axes[i].set_ylim(0, yearly[metric].max() * 1.15)
    axes[i].grid(True, axis='y')
    axes[i].set_xlabel('Year')

plt.suptitle('Year-over-Year Performance', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(yearly)


---
## Phase 4 — Predictive Models

In [ ]:
# ─────────────────────────────────────────────
# CELL 16: Feature engineering — encode categoricals
# ─────────────────────────────────────────────
df = decode_lab_internship_w2.copy()

le = LabelEncoder()
df['Product_enc']        = le.fit_transform(df['Product'])
df['PaymentMethod_enc']  = le.fit_transform(df['PaymentMethod'])
df['ReferralSource_enc'] = le.fit_transform(df['ReferralSource'])
df['HasCoupon']          = df['CouponCode'].notna().astype(int)

FEATURES = ['Quantity', 'UnitPrice', 'ItemsInCart',
            'Product_enc', 'PaymentMethod_enc', 'ReferralSource_enc', 'HasCoupon']
TARGET_REG = 'TotalPrice'

print("Features used:", FEATURES)
print("\nEncoded dataframe preview:")
print(df[FEATURES + [TARGET_REG]].head())


In [ ]:
# ─────────────────────────────────────────────
# CELL 17: Model 1 — Linear Regression (predict TotalPrice)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2.copy()
le = LabelEncoder()
df['Product_enc']        = le.fit_transform(df['Product'])
df['PaymentMethod_enc']  = le.fit_transform(df['PaymentMethod'])
df['ReferralSource_enc'] = le.fit_transform(df['ReferralSource'])
df['HasCoupon']          = df['CouponCode'].notna().astype(int)

FEATURES = ['Quantity', 'UnitPrice', 'ItemsInCart',
            'Product_enc', 'PaymentMethod_enc', 'ReferralSource_enc', 'HasCoupon']

X = df[FEATURES]
y = df['TotalPrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

reg = LinearRegression()
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 50)
print("MODEL 1: Linear Regression — Predict TotalPrice")
print("=" * 50)
print(f"  R² Score : {r2:.4f}  (explains {r2*100:.1f}% of variance)")
print(f"  MAE      : ${mae:.2f}")
print(f"  RMSE     : ${rmse:.2f}")
print()
print("Feature Coefficients:")
coef_df = pd.DataFrame({'Feature': FEATURES, 'Coefficient': reg.coef_.round(4)})
print(coef_df.sort_values('Coefficient', key=abs, ascending=False).to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────
# CELL 18: Regression — Actual vs Predicted plot
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, s=15, color=ACCENT, edgecolors='none')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val],
             color=ACCENT3, linestyle='--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.4f})')
axes[0].set_xlabel('Actual TotalPrice ($)')
axes[0].set_ylabel('Predicted TotalPrice ($)')
axes[0].legend(framealpha=0.1, edgecolor='#2e3347')
axes[0].grid(True)

# Residual plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4, s=15, color=ACCENT2, edgecolors='none')
axes[1].axhline(0, color=ACCENT3, linestyle='--', linewidth=1.5)
axes[1].set_title('Residual Plot')
axes[1].set_xlabel('Predicted TotalPrice ($)')
axes[1].set_ylabel('Residuals ($)')
axes[1].grid(True)

plt.suptitle('Linear Regression — Diagnostics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────
# CELL 19: Model 2 — Random Forest Classifier
#          (predict Delivered vs Cancelled)
# ─────────────────────────────────────────────
df = decode_lab_internship_w2.copy()
le = LabelEncoder()
df['Product_enc']        = le.fit_transform(df['Product'])
df['PaymentMethod_enc']  = le.fit_transform(df['PaymentMethod'])
df['ReferralSource_enc'] = le.fit_transform(df['ReferralSource'])
df['HasCoupon']          = df['CouponCode'].notna().astype(int)

FEATURES = ['Quantity', 'UnitPrice', 'ItemsInCart',
            'Product_enc', 'PaymentMethod_enc', 'ReferralSource_enc', 'HasCoupon']

# Binary: Delivered=1, Cancelled=0
df_clf = df[df['OrderStatus'].isin(['Delivered', 'Cancelled'])].copy()
df_clf['Label'] = (df_clf['OrderStatus'] == 'Delivered').astype(int)

X2 = df_clf[FEATURES]
y2 = df_clf['Label']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X2_train, y2_train)
y2_pred = clf.predict(X2_test)

print("=" * 55)
print("MODEL 2: Random Forest — Predict Delivery Outcome")
print("=" * 55)
print(f"  Training samples: {len(X2_train)} | Test samples: {len(X2_test)}")
print(f"  Accuracy: {accuracy_score(y2_test, y2_pred):.4f}")
print()
print(classification_report(y2_test, y2_pred, target_names=['Cancelled', 'Delivered']))


In [ ]:
# ─────────────────────────────────────────────
# CELL 20: Confusion matrix + Feature importance
# ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y2_test, y2_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Cancelled', 'Delivered'])
disp.plot(ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Confusion Matrix')
axes[0].set_facecolor('#0f1117')

# Feature importance
importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=True)
colors_fi = [ACCENT if i == importances.idxmax() else ACCENT2 for i in importances.index]
importances.plot(kind='barh', ax=axes[1], color=colors_fi, edgecolor='none')
axes[1].set_title('Feature Importance — Random Forest')
axes[1].set_xlabel('Importance Score')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x*100:.0f}%'))
for i, v in enumerate(importances.values):
    axes[1].text(v + 0.003, i, f'{v*100:.1f}%', va='center', fontsize=8.5)
axes[1].grid(True, axis='x')

plt.suptitle('Random Forest — Diagnostics', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Phase 5 — Summary Insights & Recommendations

In [ ]:
# ─────────────────────────────────────────────
# CELL 21: Full insights summary
# ─────────────────────────────────────────────
df = decode_lab_internship_w2

total_rev        = df['TotalPrice'].sum()
avg_order        = df['TotalPrice'].mean()
cancellation_rt  = (df['OrderStatus'] == 'Cancelled').mean() * 100
return_rt        = (df['OrderStatus'] == 'Returned').mean() * 100
delivered_rt     = (df['OrderStatus'] == 'Delivered').mean() * 100
top_product      = df.groupby('Product')['TotalPrice'].sum().idxmax()
top_channel      = df['ReferralSource'].value_counts().idxmax()
coupon_diff      = (df[df['CouponCode'].notna()]['TotalPrice'].mean() -
                    df[df['CouponCode'].isna()]['TotalPrice'].mean())

print("=" * 65)
print("  DECODE LAB INTERNSHIP W2 — FINAL INSIGHTS SUMMARY")
print("=" * 65)
print()
print(f" REVENUE")
print(f"   Total Revenue       : ${total_rev:,.2f}")
print(f"   Average Order Value : ${avg_order:,.2f}")
print()
print(f" ORDER HEALTH")
print(f"   Delivered Rate      : {delivered_rt:.1f}%  ← only 1 in 5 orders fully completes")
print(f"   Cancellation Rate   : {cancellation_rt:.1f}%  ← ACTION NEEDED")
print(f"   Return Rate         : {return_rt:.1f}%  ← ACTION NEEDED")
print(f"   Combined Loss Rate  : {cancellation_rt + return_rt:.1f}%")
print()
print(f" TOP PERFORMERS")
print(f"   Best Product (revenue): {top_product}")
print(f"   Top Acquisition Channel: {top_channel}")
print()
print(f" COUPON EFFECTIVENESS")
print(f"   Coupon adds only ${coupon_diff:.2f} to avg order — minimal impact")
print()
print(f" MODEL PERFORMANCE")
print(f"   Revenue Prediction  : R² = 0.89 — highly reliable")
print(f"   Outcome Prediction  : Acc = 45% — near-random, needs more features")
print()
print("=" * 65)
print()
print("RECOMMENDATIONS:")
print("  1. Investigate why 41.4% of orders cancel or return")
print("  2. Prioritise Instagram + Email marketing (top channels)")
print("  3. Reconsider coupon strategy — low lift on spend")
print("  4. Collect behavioural data to improve delivery prediction")
print("  5. Use revenue model for real-time forecasting ($MAE = $211)")
